# 03 · Data larger than RAM: pandas vs Polars vs DuckDB

**Question:** can we query all 24 months of FHVHV (2024 + 2025, ~480M trips) on a 64 GB machine,
when loading them in pandas would need ~103 GB (see `01_memory_analysis.ipynb`)?

**Query:** trips, miles, trip time, fare and driver pay per pickup month and company.

**Engines:**
- **pandas**: manual map-reduce. Read 6 of 25 columns, one file at a time, aggregate, then aggregate the partials
- **Polars**: lazy `scan_parquet` + streaming engine
- **DuckDB**: SQL straight on the Parquet files

The query code lives in `benchmarks/fhvhv_aggregate.py`. Each run starts a **new Python process**, so that:
1. peak memory is measured per engine (memory freed by one library is not always returned to the OS)
2. no engine benefits from objects another one left in memory

What is *not* isolated: the **OS file cache**. After the first read, the Parquet files are served from RAM,
not disk. That is why we repeat each run and use the median.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

DATA = Path("datasets")
OUT = Path("reports/benchmark")
SCRIPT = Path("benchmarks/fhvhv_aggregate.py")
ENGINES = ["pandas", "polars", "duckdb"]
REPEATS = 3  # pandas takes ~100 s per run: the whole notebook takes ~6 minutes


def run(engine: str, threads: int | None = None) -> dict:
    """Run the benchmark script in a fresh process and return its JSON metrics."""
    env = os.environ.copy()
    suffix = ""
    if threads:
        env["BENCH_THREADS"] = env["POLARS_MAX_THREADS"] = str(threads)
        suffix = f"_{threads}t"
    output = OUT / f"{engine}{suffix}.parquet"
    proc = subprocess.run(
        [sys.executable, str(SCRIPT), engine, str(DATA), str(output)],
        capture_output=True, text=True, env=env, check=True,
    )
    used = 1 if engine == "pandas" else (threads or os.cpu_count())  # pandas is single-threaded
    return json.loads(proc.stdout.strip().splitlines()[-1]) | {"threads": used}

## 1. Input size

In [ ]:
files = sorted(DATA.glob("fhvhv_tripdata_*.parquet"))
rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
disk_gb = sum(f.stat().st_size for f in files) / 1e9
print(f"{len(files)} files | {rows:,} rows | {disk_gb:.1f} GB on disk (compressed Parquet)")
print(f"CPU threads available: {os.cpu_count()}")

## 2. Run: all threads

Polars and DuckDB use every CPU thread by default. pandas runs on one.

In [ ]:
results = []
for i in range(REPEATS):
    for engine in ENGINES:
        r = run(engine)
        results.append(r | {"run": i + 1})
        print(r)

runs = pd.DataFrame(results)

In [ ]:
summary = (
    runs.groupby("engine")
    .agg(seconds=("seconds", "median"), peak_gb=("peak_mb", lambda s: s.max() / 1e3))
    .loc[ENGINES]
)
summary["million_rows_per_s"] = rows / summary["seconds"] / 1e6
summary["vs_pandas"] = summary.loc["pandas", "seconds"] / summary["seconds"]
summary.round(2)

## 3. Same answer?

A benchmark is only fair if every engine computes the same thing. Trip counts must match exactly.
Float sums may differ in the last digits: addition in a different order rounds differently.

In [ ]:
res = {e: pd.read_parquet(OUT / f"{e}.parquet") for e in ENGINES}
base = res["duckdb"]
SUMS = ["trip_miles", "trip_time", "base_passenger_fare", "driver_pay"]
for e in ["pandas", "polars"]:
    d = res[e]
    same_keys = d[["month", "hvfhs_license_num"]].equals(base[["month", "hvfhs_license_num"]])
    same_trips = (d["trips"].astype("int64") == base["trips"]).all()
    max_rel = max(((d[c] - base[c]).abs() / base[c].abs()).max() for c in SUMS)
    print(f"{e:7s} vs duckdb | keys equal: {same_keys} | trips equal: {same_trips} | max relative diff in sums: {max_rel:.1e}")

## 4. Run: one thread

How much of the speed comes from **parallelism** and how much from the **engine** itself
(columnar, vectorized execution, reading Parquet without building a pandas DataFrame)?
Limit Polars and DuckDB to one thread, the same as pandas.

In [ ]:
single = pd.DataFrame([run(e, threads=1) for e in ["polars", "duckdb"]])
single

In [ ]:
compare = pd.concat([
    runs.groupby("engine").agg(seconds=("seconds", "median"), peak_mb=("peak_mb", "max"), threads=("threads", "first")).reset_index(),
    single[["engine", "seconds", "peak_mb", "threads"]],
])
compare["label"] = compare["engine"] + " (" + compare["threads"].astype(str) + "t)"
compare = compare.sort_values("seconds", ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(compare["label"], compare["seconds"])
axes[0].set_title("Time (s), lower is better")
axes[1].barh(compare["label"], compare["peak_mb"] / 1e3)
axes[1].set_title("Peak memory (GB), lower is better")
for ax in axes:
    ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
compare.drop(columns="label")

## 5. The result itself: monthly trips, Uber vs Lyft

In [ ]:
COMPANY = {"HV0003": "Uber", "HV0005": "Lyft"}
monthly = base.assign(company=base["hvfhs_license_num"].map(COMPANY))
monthly["avg_fare"] = monthly["base_passenger_fare"] / monthly["trips"]
monthly["driver_share"] = monthly["driver_pay"] / monthly["base_passenger_fare"]

trips = monthly.pivot(index="month", columns="company", values="trips") / 1e6
ax = trips.plot(figsize=(12, 4), marker="o", title="Trips per month (millions)")
ax.grid(alpha=0.3)
monthly.pivot(index="month", columns="company", values=["avg_fare", "driver_share"]).round(2)

## 6. Findings

Measured on 2026-11-20: 24 threads, 64 GB RAM, 483M rows / 11.7 GB of Parquet, files already in the OS cache.
Your numbers will differ; the ratios are what matter.

| Engine | Threads | Time | Peak memory |
|---|---|---|---|
| pandas (map-reduce by hand) | 1 | ~101 s | ~3.5 GB |
| Polars (lazy + streaming) | 24 | ~2.6 s | 2.5 to 6.0 GB (varied across 6 runs) |
| DuckDB (SQL) | 24 | ~2.2 s | ~2.5 GB |
| Polars | 1 | ~44 s | ~0.7 GB |
| DuckDB | 1 | ~44 s | ~0.15 GB |

- **All three engines answer the question, with identical results.** Even pandas, because the query needs only 6 of 25
  columns and one file at a time. *Projection* (reading only the needed columns) is the first tool against data larger than RAM:
  ~103 GB in memory became ~3.5 GB
- **pandas only works because we wrote the map-reduce by hand**: loop over files, partial aggregates, aggregate again
  (a month can show up in two files). A join or a distinct count across files would be much harder. Polars and DuckDB
  express the whole query at once and plan it themselves
- **Where the ~45x comes from:** with one thread, Polars and DuckDB take ~44 s, about **2.3x** faster than pandas.
  That is the engine: vectorized, columnar execution on Arrow data. The other **~20x** is parallelism over 24 threads
- **Memory:** DuckDB with one thread peaks at **0.15 GB** for 483M rows. It *streams*: reads a batch, updates the
  running aggregate, drops the batch. More threads means more batches in flight, so more memory. DuckDB was also the most
  stable; Polars' peak varied from 2.5 to 6 GB between identical runs
- **Schema drift bites again:** Polars failed on the first try because 2025 files have `cbd_congestion_fee` and 2024 files
  do not. Fixed with `scan_parquet(..., extra_columns="ignore")`. DuckDB tolerated it; `read_parquet(..., union_by_name=true)`
  is the explicit way to merge different schemas
- **Bonus from the data:** Uber's average base fare rose from ~$24 to ~$28–29 during 2025, while Lyft's stayed around $25

**Decision for the next steps:** use **DuckDB** for local heavy lifting. It is the fastest and leanest here, and it speaks SQL,
the same language as BigQuery and dbt. Polars is an equally good choice when the code is Python rather than SQL.